# SiO2/HfO2 (001) Superlattices with Monoclinic and Cubic HfO2

## 0. Introduction

This tutorial builds two SiO₂/HfO₂ (001) superlattices and their bulk reference cells after the publications:

> **Pierre-Yves Prodhomme, Fabien Fontaine-Vive, Abram Van Der Geest, Philippe Blaise and Jacky Even**, "Ab initio
> calculation of effective work functions for a TiN/HfO2/SiO2/Si transistor stack", Applied Physics Letters 99,
> 022101 (2011). [DOI: 10.1063/1.3609869](https://doi.org/10.1063/1.3609869)

> **John Robertson and Robert M. Wallace**, "High-K materials and metal gates for CMOS applications", Materials
> Science and Engineering: R: Reports 88, 1 (2015). [DOI: 10.1016/j.mser.2014.11.001](https://doi.org/10.1016/j.mser.2014.11.001)

SiO₂ is ideal β-cristobalite (C9) and is the in-plane reference: its (001) face is the 5.23 × 5.23 Å² square cell,
turned 45° from the cubic cell, and HfO₂ is strained onto it. Both superlattices are periodic along [001] without
vacuum, so each has two SiO₂/HfO₂ interfaces.

- **Monoclinic HfO₂** (P2₁/c), as in the first publication: ten SiO₂ layers (one Si each) and seven HfO₂ layers (two
  Hf each), Hf₁₄Si₁₀O₄₈. Each interface carries three O between the last Si and the first Hf. The publication
  stretches HfO₂ by 3.7 % along [100] and 2.1 % along [010] from its own lattice; here the strain follows from the
  Standata lattices.
- **Cubic HfO₂** (Fm-3m), the model of the second publication, built after Sharia et al. [3]: twelve SiO₂ and six
  HfO₂ layers, Hf₁₂Si₁₂O₄₈. Each interface plane carries three O: two bridge Si and Hf, one is bonded to Hf only, so
  no Si–O bond is broken.

At each interface the two blocks are separated so that the closest cation–O pair across it is at the bulk bond length
of the cation's own oxide; both publications relax the atomic positions afterwards.

The valence band offsets are computed from the superlattices and from bulk cells with the same in-plane lattice, so
all five materials are saved:
[the simulation notebook](interface_3d_3d_silicon_dioxide_hafnium_dioxide_SIMULATION.ipynb) loads them by name.

## 1. Prepare the Environment
### 1.1. Set up the parameters

In [ ]:
SUBSTRATE_NAME = "mp-8352"  # SiO2, beta-cristobalite (C9), Standata
MONOCLINIC_FILM_NAME = "mp-352"  # HfO2, monoclinic, Standata
CUBIC_FILM_NAME = "mp-550893"  # HfO2, cubic, Standata

SUBSTRATE_MILLER_INDICES = (1, 1, 0)  # in the primitive cell: the (001) face, 5.23 Å square
FILM_MILLER_INDICES = (0, 0, 1)
VACUUM = 0.0  # in Angstrom: periodic along the stacking axis

MONOCLINIC_SUBSTRATE_NUMBER_OF_LAYERS = 5  # two SiO2 layers each: 10
MONOCLINIC_FILM_NUMBER_OF_LAYERS = 4  # two HfO2 layers each: 8, of which one is removed
MONOCLINIC_FILM_ATOMS_REMOVED_FROM_BOTTOM = 1  # the lowest O
MONOCLINIC_FILM_ATOMS_REMOVED_FROM_TOP = 5  # with the lowest O, one HfO2 layer; three O stay at each interface

CUBIC_SUBSTRATE_NUMBER_OF_LAYERS = 6  # two SiO2 layers each: 12
CUBIC_FILM_NUMBER_OF_LAYERS = 3  # two HfO2 layers each: 6
CUBIC_FILM_XY_SHIFT = (0, 0.25)  # fractions of the in-plane cell: SiO2 interface O on HfO2 O sites

SUBSTRATE_BULK_NUMBER_OF_LAYERS = 2  # one beta-cristobalite period along [001]

MAX_LENGTH_TOLERANCE = 0.05  # admits the 4 % mismatch of cubic HfO2

MONOCLINIC_SUPERLATTICE_NAME = "SiO2/HfO2 monoclinic (001) superlattice 10+7"
CUBIC_SUPERLATTICE_NAME = "SiO2/HfO2 cubic (001) superlattice 12+6"
SUBSTRATE_BULK_NAME = "SiO2 beta-cristobalite (001) bulk"
MONOCLINIC_FILM_BULK_NAME = "HfO2 monoclinic (001) bulk strained"
CUBIC_FILM_BULK_NAME = "HfO2 cubic (001) bulk strained"

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Load the materials and define the helper functions

The SiO₂ slab is cut from the primitive cell, where the Miller indices (1, 1, 0) select the cubic (001) face in the
45° cell. The interface builder re-creates its slabs from the bulk, so layer counts and atoms are changed on the built
superlattice. At each interface the film is moved along the normal, and the cell length is set, so that the closest
cation–O pair across the interface is at the bulk bond length: Si–O of SiO₂, Hf–O of the HfO₂ phase.

In [ ]:
import numpy as np
from ase.geometry import get_distances
from mat3ra.esse.models.core.abstract.matrix_3x3 import Matrix3x3Schema
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import to_ase
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.made.tools.helpers import (create_interface_zsl_between_slabs, create_slab,
                                       get_material_with_conventional_lattice)
from mat3ra.made.tools.modify import interface_displace_part, interface_get_part
from mat3ra.made.tools.operations.core.unary import strain
from mat3ra.made.utils import adjust_material_cell_to_set_gap_along_direction
from mat3ra.standata.materials import Materials


def get_conventional_cell(name):
    return get_material_with_conventional_lattice(Material.create(Materials.get_by_name_first_match(name)))


def get_bond_length(material, cation):
    atoms = to_ase(material)
    symbols = np.array(atoms.get_chemical_symbols())
    return atoms.get_all_distances(mic=True)[np.ix_(symbols == cation, symbols == "O")].min()


def get_vertical_shift(lower, upper, bond_length_by_pair, upper_offset=0.0):
    """Shift along z of `upper` that puts the closest cation–O pair across the interface at its bond length."""
    lower_atoms, upper_atoms = to_ase(lower), to_ase(upper)
    vectors, _ = get_distances(lower_atoms.positions, upper_atoms.positions + [0, 0, upper_offset],
                               cell=lower_atoms.cell, pbc=[True, True, False])
    bond_length = np.array([[bond_length_by_pair.get(frozenset((lower_symbol, upper_symbol)), 0.0)
                             for upper_symbol in upper_atoms.symbols] for lower_symbol in lower_atoms.symbols])
    in_plane_distance = np.linalg.norm(vectors[..., :2], axis=-1)
    paired = bond_length > in_plane_distance
    return (np.sqrt(bond_length[paired] ** 2 - in_plane_distance[paired] ** 2) - vectors[..., 2][paired]).max()


def set_interface_separations(interface, film):
    bond_length_by_pair = {frozenset(("Si", "O")): get_bond_length(substrate, "Si"),
                           frozenset(("Hf", "O")): get_bond_length(film, "Hf")}
    substrate_part = interface_get_part(interface, InterfacePartsEnum.SUBSTRATE)
    film_part = interface_get_part(interface, InterfacePartsEnum.FILM)
    film_shift = get_vertical_shift(substrate_part, film_part, bond_length_by_pair)
    wrap_shift = get_vertical_shift(film_part, substrate_part, bond_length_by_pair, interface.lattice.c)
    top = to_ase(interface).positions[:, 2].max()
    interface = adjust_material_cell_to_set_gap_along_direction(interface,
                                                                interface.lattice.c + film_shift + wrap_shift - top)
    return interface_displace_part(interface, [0, 0, film_shift])


def strain_in_plane(material, lattice):
    vectors = np.array(material.lattice.vector_arrays)
    a_direction, b_direction = vectors[:2] / np.linalg.norm(vectors[:2], axis=1)[:, None]
    normal = np.cross(a_direction, b_direction)
    deformation = (lattice.a / material.lattice.a * np.outer(a_direction, a_direction)
                   + lattice.b / material.lattice.b * np.outer(b_direction, b_direction) + np.outer(normal, normal))
    return strain(material, Matrix3x3Schema(root=deformation.tolist()))


def create_superlattice(film, substrate_number_of_layers, film_number_of_layers, xy_shift=(0, 0)):
    substrate_slab = create_slab(substrate, SUBSTRATE_MILLER_INDICES, use_conventional_cell=False,
                                 number_of_layers=substrate_number_of_layers, vacuum=VACUUM)
    film_slab = create_slab(film, FILM_MILLER_INDICES, number_of_layers=film_number_of_layers, vacuum=VACUUM)
    return create_interface_zsl_between_slabs(
        substrate_slab=substrate_slab, film_slab=film_slab, vacuum=VACUUM, max_length_tol=MAX_LENGTH_TOLERANCE,
        xy_shift=(np.array(xy_shift) * [substrate_slab.lattice.a, substrate_slab.lattice.b]).tolist())


substrate = Material.create(Materials.get_by_name_first_match(SUBSTRATE_NAME))
monoclinic_film = get_conventional_cell(MONOCLINIC_FILM_NAME)
cubic_film = get_conventional_cell(CUBIC_FILM_NAME)

## 2. Create the Superlattice with Monoclinic HfO₂

The slab builder counts monoclinic (001) layers in pairs, so eight HfO₂ layers are built and one is removed. The
HfO₂ block as built ends in two O on each side; the block kept is one O higher, so it ends in one O against the
SiO₂ O plane and in three O against the SiO₂ Si. Each interface then carries three O, and the cell stays
stoichiometric. The in-plane registry of the two blocks is not given in the publication; the builder's default is
kept.

In [ ]:
from mat3ra.made.tools.modify import filter_by_ids

monoclinic_superlattice = create_superlattice(monoclinic_film, MONOCLINIC_SUBSTRATE_NUMBER_OF_LAYERS,
                                              MONOCLINIC_FILM_NUMBER_OF_LAYERS)
heights = to_ase(monoclinic_superlattice).positions[:, 2]
film_ids = np.flatnonzero(np.array(monoclinic_superlattice.basis.labels.values) == InterfacePartsEnum.FILM.value)
film_ids_by_height = film_ids[np.argsort(heights[film_ids])]
removed_ids = np.concatenate([film_ids_by_height[:MONOCLINIC_FILM_ATOMS_REMOVED_FROM_BOTTOM],
                              film_ids_by_height[-MONOCLINIC_FILM_ATOMS_REMOVED_FROM_TOP:]])
monoclinic_superlattice = filter_by_ids(monoclinic_superlattice, removed_ids.tolist(), invert=True, reset_ids=True)
monoclinic_superlattice = set_interface_separations(monoclinic_superlattice, monoclinic_film)

## 3. Create the Superlattice with Cubic HfO₂
### 3.1. Stack the slabs

The HfO₂ film is shifted in plane by a quarter of the cell, so that the O of SiO₂ at the interface sit on O sites of
HfO₂. As built, one interface plane is an HfO₂ O plane with four O, two of them bonded to the Si above, and the other
is the SiO₂ O plane with two O.

In [ ]:
cubic_superlattice = create_superlattice(cubic_film, CUBIC_SUBSTRATE_NUMBER_OF_LAYERS, CUBIC_FILM_NUMBER_OF_LAYERS,
                                         CUBIC_FILM_XY_SHIFT)
cubic_superlattice = set_interface_separations(cubic_superlattice, cubic_film)

### 3.2. Move one O to the other interface

The first O of the four-O plane that is not bonded to Si is removed, and an O is added at the same in-plane position
in the two-O plane. Each interface plane then carries three O, two bonded to Si and Hf and one to Hf only: the
structure of Sharia et al., who remove a different O from each of two four-O interface planes.

In [ ]:
from mat3ra.made.tools.helpers import create_defect_point_interstitial, create_defect_point_vacancy

atoms = to_ase(cubic_superlattice)
symbols = np.array(atoms.get_chemical_symbols())
labels = np.array(cubic_superlattice.basis.labels.values)
heights = atoms.positions[:, 2]
film_oxygen_ids = np.flatnonzero((symbols == "O") & (labels == InterfacePartsEnum.FILM.value))
substrate_oxygen_ids = np.flatnonzero((symbols == "O") & (labels == InterfacePartsEnum.SUBSTRATE.value))
four_oxygen_plane_ids = film_oxygen_ids[np.isclose(heights[film_oxygen_ids], heights[film_oxygen_ids].max())]
distance_to_silicon = [atoms.get_distances(index, np.flatnonzero(symbols == "Si"), mic=True).min()
                       for index in four_oxygen_plane_ids]
removed_position = atoms.positions[four_oxygen_plane_ids[np.isclose(distance_to_silicon, max(distance_to_silicon))][0]]
added_position = [*removed_position[:2], heights[substrate_oxygen_ids].max()]

cubic_superlattice = create_defect_point_vacancy(cubic_superlattice, removed_position.tolist(), "closest_site",
                                                 use_cartesian_coordinates=True)
cubic_superlattice = create_defect_point_interstitial(cubic_superlattice, added_position, "O", "exact_coordinate",
                                                      use_cartesian_coordinates=True)

## 4. Create the Bulk Reference Cells

SiO₂ is used as is, one β-cristobalite period along [001] in the 45° cell. Each HfO₂ is strained in plane to the cell
of its superlattice, with the (001) layer spacing kept. Atom labels are removed from all five materials: labels break
the Quantum ESPRESSO input.

In [ ]:
substrate_bulk = create_slab(substrate, SUBSTRATE_MILLER_INDICES, use_conventional_cell=False,
                             number_of_layers=SUBSTRATE_BULK_NUMBER_OF_LAYERS, vacuum=VACUUM)
monoclinic_film_bulk = strain_in_plane(monoclinic_film, monoclinic_superlattice.lattice)
cubic_film_bulk = strain_in_plane(cubic_film, cubic_superlattice.lattice)

materials = [monoclinic_superlattice, cubic_superlattice, substrate_bulk, monoclinic_film_bulk, cubic_film_bulk]
names = [MONOCLINIC_SUPERLATTICE_NAME, CUBIC_SUPERLATTICE_NAME, SUBSTRATE_BULK_NAME, MONOCLINIC_FILM_BULK_NAME,
         CUBIC_FILM_BULK_NAME]
for material, name in zip(materials, names):
    material.name = name
    material.set_labels_from_list([])

## 5. Visualize the Result(s)

In each superlattice the SiO₂ block is at the bottom of the cell and the HfO₂ block on top; the second interface is
across the periodic boundary.

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials([{"material": material, "title": material.name, "rotation": "-90x"} for material in materials],
                    repetitions=[2, 2, 1])

## 6. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

set_materials(materials)

## References

[1] P.-Y. Prodhomme, F. Fontaine-Vive, A. Van Der Geest, P. Blaise and J. Even, "Ab initio calculation of effective
work functions for a TiN/HfO2/SiO2/Si transistor stack", Appl. Phys. Lett. 99, 022101 (2011).
[DOI: 10.1063/1.3609869](https://doi.org/10.1063/1.3609869)

[2] J. Robertson and R. M. Wallace, "High-K materials and metal gates for CMOS applications", Mater. Sci. Eng. R 88,
1 (2015). [DOI: 10.1016/j.mser.2014.11.001](https://doi.org/10.1016/j.mser.2014.11.001)

[3] O. Sharia, A. A. Demkov, G. Bersuker and B. H. Lee, "Theoretical study of the insulator/insulator interface: Band
alignment at the SiO2/HfO2 junction", Phys. Rev. B 75, 035306 (2007).
[DOI: 10.1103/PhysRevB.75.035306](https://doi.org/10.1103/PhysRevB.75.035306)